In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/adityachandra1611/sample-submission/sample_submission.csv
/kaggle/input/datasets/adityachandra1611/train-dataset/train.csv
/kaggle/input/datasets/adityachandra1611/test-traffic/test.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor

In [3]:
train = pd.read_csv("/kaggle/input/datasets/adityachandra1611/train-dataset/train.csv")
test = pd.read_csv("/kaggle/input/datasets/adityachandra1611/test-traffic/test.csv")

target = "demand"

X = train.drop(columns=[target])
y = train[target]

X_test = test.copy()

print(X.shape)
print(y.shape)
print(X_test.shape)

(77299, 10)
(77299,)
(41778, 10)


In [4]:
print(train.columns.tolist())
print(train.head())

['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']
   Index geohash  day timestamp    demand     RoadType  NumberofLanes  \
0      0  qp02z1   48       0:0  0.048804          NaN              1   
1      1  qp02zt   48       0:0  0.118507  Residential              3   
2      2  qp08bj   48       0:0  0.027132  Residential              1   
3      3  qp08gt   48       0:0  0.003272  Residential              1   
4      4  qp02zq   48       0:0  0.010819  Residential              1   

  LargeVehicles Landmarks  Temperature Weather  
0   Not Allowed        No          NaN     NaN  
1       Allowed       Yes    31.104565   Sunny  
2   Not Allowed        No    25.919267   Sunny  
3   Not Allowed        No          NaN   Rainy  
4   Not Allowed        No    10.803667   Rainy  


In [5]:
train.drop(
    columns=[
        'year',
        'month',
        'hour',
        'minute',
        'weekofyear'
    ],
    inplace=True,
    errors='ignore'
)

test.drop(
    columns=[
        'year',
        'month',
        'hour',
        'minute',
        'weekofyear'
    ],
    inplace=True,
    errors='ignore'
)

In [6]:
target = 'demand'

X = train.drop(columns=[target])

y = train[target]

X_test = test.copy()

In [7]:
for col in X.columns:

    if X[col].dtype == 'object':

        X[col] = X[col].fillna('Unknown')
        X_test[col] = X_test[col].fillna('Unknown')

    else:

        X[col] = X[col].fillna(X[col].median())
        X_test[col] = X_test[col].fillna(X[col].median())

In [8]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(include='object').columns

print(cat_cols)

Index(['geohash', 'timestamp', 'RoadType', 'LargeVehicles', 'Landmarks',
       'Weather'],
      dtype='object')


In [9]:
for col in cat_cols:

    le = LabelEncoder()

    combined = pd.concat([
        X[col].astype(str),
        X_test[col].astype(str)
    ])

    le.fit(combined)

    X[col] = le.transform(X[col].astype(str))

    X_test[col] = le.transform(
        X_test[col].astype(str)
    )

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=2000,
    depth=8,
    learning_rate=0.03,
    loss_function='RMSE',
    verbose=100
)

model.fit(X_train, y_train)

pred = model.predict(X_valid)

print(
    "R2 Score:",
    r2_score(y_valid, pred)
)

0:	learn: 0.1389468	total: 65.7ms	remaining: 2m 11s
100:	learn: 0.0650841	total: 659ms	remaining: 12.4s
200:	learn: 0.0612273	total: 1.34s	remaining: 12s
300:	learn: 0.0590533	total: 2s	remaining: 11.3s
400:	learn: 0.0571128	total: 2.59s	remaining: 10.3s
500:	learn: 0.0556200	total: 3.19s	remaining: 9.54s
600:	learn: 0.0544861	total: 3.83s	remaining: 8.93s
700:	learn: 0.0535982	total: 4.47s	remaining: 8.28s
800:	learn: 0.0528277	total: 5.04s	remaining: 7.54s
900:	learn: 0.0521612	total: 5.66s	remaining: 6.9s
1000:	learn: 0.0515245	total: 6.3s	remaining: 6.29s
1100:	learn: 0.0508111	total: 6.92s	remaining: 5.65s
1200:	learn: 0.0502920	total: 7.47s	remaining: 4.97s
1300:	learn: 0.0498362	total: 8.09s	remaining: 4.35s
1400:	learn: 0.0493708	total: 8.73s	remaining: 3.73s
1500:	learn: 0.0489892	total: 9.36s	remaining: 3.11s
1600:	learn: 0.0485918	total: 10s	remaining: 2.49s
1700:	learn: 0.0482403	total: 10.6s	remaining: 1.87s
1800:	learn: 0.0478955	total: 11.3s	remaining: 1.25s
1900:	learn:

In [11]:
print(X.shape)
print(X_test.shape)
print(X.isnull().sum().sum())

(77299, 10)
(41778, 10)
0


In [12]:
CatBoost : 0.50
LightGBM : 0.25
XGBoost : 0.15
ExtraTrees : 0.10

In [13]:
print(X.columns.tolist())

['Index', 'geohash', 'day', 'timestamp', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']


In [14]:
from sklearn.model_selection import KFold

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [15]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor

In [16]:
cat_model = CatBoostRegressor(
    iterations=1500,
    depth=8,
    learning_rate=0.03,
    verbose=0
)

lgb_model = LGBMRegressor(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=8,
    random_state=42
)

xgb_model = XGBRegressor(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=8,
    random_state=42,
    tree_method="hist"
)

et_model = ExtraTreesRegressor(
    n_estimators=400,
    random_state=42,
    n_jobs=-1
)

In [17]:
import numpy as np

pred_cat = np.zeros(len(X_test))
pred_lgb = np.zeros(len(X_test))
pred_xgb = np.zeros(len(X_test))
pred_et  = np.zeros(len(X_test))

cv_scores = []

In [18]:
#from sklearn.metrics import r2_score

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

    print(f"\nFold {fold+1}")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # CatBoost
    cat_model.fit(X_train, y_train)

    val_cat = cat_model.predict(X_val)

    pred_cat += cat_model.predict(X_test) / 5

    # LightGBM
    lgb_model.fit(X_train, y_train)

    val_lgb = lgb_model.predict(X_val)

    pred_lgb += lgb_model.predict(X_test) / 5

    # XGBoost
    xgb_model.fit(X_train, y_train)

    val_xgb = xgb_model.predict(X_val)

    pred_xgb += xgb_model.predict(X_test) / 5

    # ExtraTrees
    et_model.fit(X_train, y_train)

    val_et = et_model.predict(X_val)

    pred_et += et_model.predict(X_test) / 5

    ensemble_pred = (
        0.50 * val_cat +
        0.25 * val_lgb +
        0.15 * val_xgb +
        0.10 * val_et
    )

    score = r2_score(y_val, ensemble_pred)

    cv_scores.append(score)

    print("Fold Score =", score) 


Fold 1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005202 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 883
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 10
[LightGBM] [Info] Start training from score 0.093784
Fold Score = 0.8728522508412847

Fold 2
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002339 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 883
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 10
[LightGBM] [Info] Start training from score 0.093797
Fold Score = 0.8721128118730463

Fold 3
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002391 seconds

In [19]:
print("Mean CV =", np.mean(cv_scores))

Mean CV = 0.8715427249375459


In [20]:
X = X.drop(columns=['Index'])
X_test = X_test.drop(columns=['Index'])

In [21]:
print("Cat :", r2_score(y_val, val_cat))
print("LGB :", r2_score(y_val, val_lgb))
print("XGB :", r2_score(y_val, val_xgb))
print("ET  :", r2_score(y_val, val_et))

Cat : 0.8579863409072797
LGB : 0.8598617972828285
XGB : 0.8597221117601104
ET  : 0.8896002450414205


In [22]:
ensemble_pred = (
    0.60 * val_cat +
    0.25 * val_lgb +
    0.10 * val_xgb +
    0.05 * val_et
)

In [23]:
pred_cat = np.zeros(len(X_test))
scores = []

for train_idx, val_idx in kf.split(X):

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    cat_model.fit(X_train, y_train)

    val_pred = cat_model.predict(X_val)

    scores.append(r2_score(y_val, val_pred))

    pred_cat += cat_model.predict(X_test)/5

print(np.mean(scores))

0.8527527729932916


In [24]:
print(len(pred_cat))
print(len(pred_lgb))
print(len(pred_xgb))
print(len(pred_et))

41778
41778
41778
41778


In [25]:
final_predictions = (
      0.50 * pred_cat
    + 0.25 * pred_lgb
    + 0.15 * pred_xgb
    + 0.10 * pred_et
)

In [26]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": final_predictions
})

submission.to_csv("submission.csv", index=False)

In [27]:
print(final_predictions[:5])

[0.06953019 0.02299639 0.028523   0.02358565 0.04068887]


In [28]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": final_predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())

   Index    demand
0      0  0.069530
1      1  0.022996
2      2  0.028523
3      3  0.023586
4      4  0.040689


In [29]:
import os
print(os.path.exists("submission.csv"))

True


In [30]:
sub = pd.read_csv("submission.csv")
print(sub.shape)
print(sub.head())

(41778, 2)
   Index    demand
0      0  0.069530
1      1  0.022996
2      2  0.028523
3      3  0.023586
4      4  0.040689
